# 02 Indian Option Data Pipeline

Demonstrate tolerant NSE-style CSV loading, column normalization, missing value handling, expiry parsing, and processed export.


In [ ]:
import json
import math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%run 00_project_setup_and_shared_functions.ipynb

config = load_config()
rng = set_global_seed(int(config["random_seed"]))
print("Config loaded and deterministic seed set.")


In [ ]:
raw_csvs = sorted(RAW_DATA_DIR.glob("*.csv"))
if raw_csvs:
    input_path = raw_csvs[0]
    print(f"Loading user-provided CSV: {input_path}")
    chain = load_option_chain_csv(input_path)
else:
    print("No user CSV found. Creating a clearly synthetic NSE-style sample in data/raw.")
    synthetic = generate_synthetic_indian_market(config).head(80).copy()
    sample = synthetic.rename(columns={
        "symbol": "SYMBOL", "expiry": "EXPIRY", "strike": "STRIKE_PRICE", "option_type": "OPTION_TYPE",
        "underlying_price": "UNDERLYING_VALUE", "last_price": "LTP", "implied_volatility": "IV", "open_interest": "OI",
    })
    input_path = RAW_DATA_DIR / "sample_nse_style_synthetic_option_chain.csv"
    sample.to_csv(input_path, index=False)
    chain = load_option_chain_csv(input_path)
processed_path = PROCESSED_DATA_DIR / "normalized_option_chain.csv"
chain.to_csv(processed_path, index=False)
print(f"Processed CSV written to {processed_path}")
chain.head()


In [ ]:
required = {"symbol", "expiry", "strike", "option_type", "underlying_price", "mid_price", "implied_volatility", "maturity"}
assert required.issubset(chain.columns)
assert chain["option_type"].isin(["call", "put"]).all()
assert chain["strike"].notna().any()
print("VALIDATION PASSED: CSV normalized with expected canonical columns")
summary = chain.groupby(["symbol", "option_type"]).agg(rows=("strike", "size"), min_strike=("strike", "min"), max_strike=("strike", "max"), avg_iv=("implied_volatility", "mean")).reset_index()
save_table(summary, "02_option_chain_pipeline_summary.csv")
save_output({"processed_path": str(processed_path), "rows": int(len(chain))}, "02_pipeline_output.json")
summary
